# **Лабораторная работа №3**

1) Используйте датасет Fashion-MNIST из torchvision.datasets.

Требования:
* Загружать train/test через torchvision.datasets.FashionMNIST
* Преобразовать картинки в тензоры (ToTensor())
* Сделать DataLoader:

batch_size = 128

shuffle=True только для train
* Разделить train на train/val (например 55k/5k) случайно, но воспроизводимо (torch.manual_seed(42))


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# установка seed для воспроизводимости
torch.manual_seed(42)

# трансформации: только перевод в тензор
transform = transforms.Compose([transforms.ToTensor()])

# загрузка датасетов
full_train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# разделение train на train/val (55k/5k)
train_subset, val_subset = random_split(full_train_data, [55000, 5000])

# создание DataLoader
train_loader = DataLoader(train_subset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

100%|██████████| 26.4M/26.4M [00:01<00:00, 14.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 267kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.95MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.94MB/s]




---



2) Модель (только nn.Module)

Реализуйте класс FCN(nn.Module):

Архитектура:
* вход: 28×28 → 784
* Linear(784, 256) + ReLU
* Linear(256, 128) + ReLU
* Linear(128, 10)


In [ ]:
import torch.nn as nn

class FCN(nn.Module):
    def __init__(self):
        super(FCN, self).__init__()
        self.flatten = nn.Flatten()
        self.main = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.main(x)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FCN().to(DEVICE)



---



3) Обучение

Реализуйте обучение на DEVICE (CPU/GPU):
* criterion = nn.CrossEntropyLoss()
* optimizer = Adam(model.parameters(), lr=1e-3)
* 10 эпох (или early stopping по val accuracy)

Логирование:
* печатать loss каждые 200 батчей
* после каждой эпохи считать val_accuracy


4) Сохранение лучшей модели

Если val_accuracy улучшилась — сохраняйте веса:

torch.save(model.state_dict(), "best_fashion_fcn.pth")


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 10
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # печать loss каждые 200 батчей
        if (i + 1) % 200 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(train_loader)}], Loss: {running_loss/200:.4f}")
            running_loss = 0.0

    # расчет val_accuracy после каждой эпохи
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = 100 * correct / total
    print(f"Validation Accuracy: {val_accuracy:.2f}%")

    # сохранение лучшей модели
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        torch.save(model.state_dict(), "best_fashion_fcn.pth")
        print("New best model saved!")

Epoch [1/10], Step [200/430], Loss: 0.7289
Epoch [1/10], Step [400/430], Loss: 0.4573
Validation Accuracy: 83.88%
New best model saved!
Epoch [2/10], Step [200/430], Loss: 0.4085
Epoch [2/10], Step [400/430], Loss: 0.3784
Validation Accuracy: 86.30%
New best model saved!
Epoch [3/10], Step [200/430], Loss: 0.3418
Epoch [3/10], Step [400/430], Loss: 0.3454
Validation Accuracy: 85.70%
Epoch [4/10], Step [200/430], Loss: 0.3248
Epoch [4/10], Step [400/430], Loss: 0.3091
Validation Accuracy: 87.16%
New best model saved!
Epoch [5/10], Step [200/430], Loss: 0.3040
Epoch [5/10], Step [400/430], Loss: 0.2978
Validation Accuracy: 87.94%
New best model saved!
Epoch [6/10], Step [200/430], Loss: 0.2834
Epoch [6/10], Step [400/430], Loss: 0.2870
Validation Accuracy: 87.82%
Epoch [7/10], Step [200/430], Loss: 0.2735
Epoch [7/10], Step [400/430], Loss: 0.2719
Validation Accuracy: 86.92%
Epoch [8/10], Step [200/430], Loss: 0.2637
Epoch [8/10], Step [400/430], Loss: 0.2541
Validation Accuracy: 88.06%




---



5) Тестирование

После обучения:
* загрузите сохранённые лучшие веса
* посчитайте test_accuracy через model.eval() и torch.no_grad()


Требование к результату
* test_accuracy не ниже 85% (если не получилось — в отчёте объяснить, что пробовали улучшать)


In [ ]:
# загрузка весов
model.load_state_dict(torch.load("best_fashion_fcn.pth"))
model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

test_accuracy = 100 * test_correct / test_total
print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")

if test_accuracy >= 85:
    print("Результат соответствует требованиям (>85%).")
else:
    print("Результат ниже 85% :(")


Final Test Accuracy: 88.22%
Результат соответствует требованиям (>85%).




---



# Вывод

В ходе работы была реализована и обучена полносвязная нейронная сеть (FCN) для классификации изображений Fashion-MNIST.
* Подготовлены загрузчики DataLoader с разделением на train/val (55k/5k) и фиксацией seed=42, что обеспечило стабильность обучения.
* Архитектура: Использована трехслойная модель (784-256-128-10) с активацией ReLU. Слой nn.Flatten позволил корректно подать изображения $28 \times 28$ на вход сети.
* Обучение: Применение оптимизатора Adam ($lr=10^{-3}$) и функции потерь CrossEntropyLoss позволило достичь быстрой сходимости за 10 эпох.
* Итоговая точность на тесте составила 88.22%, что выше целевого порога в 85%. Модель показала высокую эффективность в распознавании базовых категорий одежды.

Модель успешно справляется с задачей.